In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
# Add the parent of the current script's parent to sys.path
project_root = Path('./').resolve().parents[1]

In [ ]:
DATA_ROOT = Path(os.environ["EPHEMERAL"]) / "pyspoc_synthetic_data"

In [ ]:
N = 1000
P = 10

meta_df = pd.read_csv(DATA_ROOT / "manifest.csv")

syn_datasets_df = meta_df[(meta_df['n_rows']==N) & (meta_df['n_cols']==P)]

In [ ]:
syn_datasets_df

In [ ]:
from pyspoc import Config, Calculator

In [ ]:
new_mod_path_dist = os.path.join(project_root,'summary_stats','distance_distrib_stats.py')

my_config_yaml = f"""
ReducedStatistics:
    {new_mod_path_dist}:
        CentroidDistanceDistributionCompare:
            schemes:
                minkowski_distrib_meancentroid:
                    metric: minkowski
                    minkowski_p: 0.1
                    modes:
                        - WASSERSTEIN_DIST
                        - KS_STATISTIC
                        - KL_BOTH
        GlobalCentroidDiffs:
            schemes:
                minkowski_distrib_meancentroid:
                    metric: minkowski
                    minkowski_p: 0.1
        DistanceDistributionBasicSummarize:
            schemes:
                cosine_distrib_meancentroid:
                    metric: cosine
                    point: global_mean_centroid
                    stats:
                        - kurtosis
                        - skew
                        - mean
                        - median                  
                    minkowski_p: None
        DistanceDistributionModesSummarize:
            schemes:
                cosine_distrib_meancentroid:
                    metric: cosine
                    point: global_mean_centroid
                    minkowski_p: None
"""

my_config_name = "new_config1"

In [ ]:
# syn_datasets_df

In [ ]:
# n_chunks = 10
# tot_v = 0
# for i in range(n_chunks):
#     print(syn_datasets_df.iloc[i :: n_chunks].shape)
#     tot_v +=syn_datasets_df.iloc[i :: n_chunks].shape[0]
# tot_v

In [ ]:
# /rds/general/user/hpj22/home/hcda_pyspoc_project/py-spoc/working_analyses/hjResearch_2025/hpc_pyspoc_scripts/configs/test_config.yml

In [ ]:
calcs_lst = []
for ix,v in enumerate(syn_datasets_df['dataset_id']):
    # print(ix,v)    
    file_nm = v + '.npy'
    sample_X = np.load(DATA_ROOT / file_nm)
    
    cfg = Config.from_yaml(my_config_name, my_config_yaml)    
    calc = Calculator(dataset=sample_X,name=v)
    calc.compute(cfg)

    calcs_lst.append(calc)

In [ ]:
results_lst = [calcs.get_results() for calcs in calcs_lst]

In [ ]:
errors_lst = [calcs.get_errors() for calcs in calcs_lst]

In [ ]:
errors = {k: repr(e) for k, e in errors_lst[0].items() if e}
errors
if errors:
    print('c')

In [ ]:
Calculator.get_results?

In [ ]:
data_matrix = pd.concat(results_lst)

In [ ]:
data_matrix